In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv('D:\\New folder\\cars_unseen_data.csv')

In [3]:

# Step 1: Fix formatting issues
# Price (`pu`)
data['pu'] = data['pu'].str.replace(',', '').astype(float)

# Max Power
data['Max Power'] = data['Max Power'].str.extract(r'(\d+\.?\d*)bhp').astype(float)

# Max Torque
data['Max Torque'] = data['Max Torque'].str.extract(r'(\d+\.?\d*)Nm').astype(float)

# Top Speed
data['Top Speed'] = data['Top Speed'].str.extract(r'(\d+\.?\d*)').astype(float)

# Acceleration
data['Acceleration'] = data['Acceleration'].str.extract(r'(\d+\.?\d*)').astype(float)

# Mileage (`mileage_new`)
data['mileage_new'] = data['mileage_new'].str.extract(r'(\d+\.?\d*)').astype(float)

#Feature Engineering
# Convert `myear` to car age
data["car_age"] = 2025 - data["myear"]


In [4]:
# Turbo Charger and Super Charger
data['Turbo Charger'] = data['Turbo Charger'].str.lower().map({'yes': 1, 'no': 0})
data['Super Charger'] = data['Super Charger'].str.lower().map({'yes': 1, 'no': 0})

# One-hot encode 'carType' and drop the original column
if "ft" in data.columns:
    data = pd.get_dummies(data, columns=["ft"], prefix="ft")

# Clean the Gear Number column
def clean_gear_number(gear):
    if pd.isna(gear):  # Handle missing values
        return np.nan
    gear = gear.strip().lower()  # Normalize case and remove extra spaces
    if 'cvt' in gear:  # Handle CVT as null
        return np.nan
    # Extract the numeric part using regex
    numeric_part = ''.join(filter(str.isdigit, gear))
    return numeric_part if numeric_part else np.nan  # Return NaN if no number is found

# Apply the cleaning function
data['Gear Box'] = data['Gear Box'].apply(clean_gear_number)


In [5]:
data["pu"] = np.log1p(data["pu"])  # log(1 + x) for stability

In [6]:

features = ["pu", "Max Power", "Max Torque", "Top Speed", "Acceleration", "mileage_new", "Turbo Charger",
             "Super Charger", "km_driven", 'car_age' , 'ft_CNG',
            'ft_Diesel', 'ft_Electric', 'ft_LPG', 'ft_Petrol', "Gear Box"]

In [7]:
X = data[features]
y = data['tt']
# Encode target variable (`tt`)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [8]:
with open('scaler.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)

with open('imputer.pkl', 'rb') as file:
    loaded_imputer = pickle.load(file)


In [9]:

X = loaded_scaler.transform(X)

In [10]:

X = loaded_imputer.transform(X)

In [11]:
from tensorflow import keras 
loaded_model = keras.models.load_model('trained_model.keras')
loaded_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,429 (154.02 KB)

 Trainable params: 12,993 (50.75 KB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 25,988 (101.52 KB)

In [12]:
predictions = loaded_model.predict(X)

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [13]:

# Evaluate
#test_loss, test_accuracy = loaded_model.evaluate(X, y, verbose=0)
#print(f"Validation Accuracy: {test_accuracy}")
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate the model (returns loss and accuracy)
test_loss, test_accuracy = loaded_model.evaluate(X, y, verbose=0)

# Predict probabilities and convert to binary labels
y_pred_prob = loaded_model.predict(X)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()  # Flatten to ensure 1D array

# Calculate metrics
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)

# Print results
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step
Test Loss: 0.1593
Test Accuracy: 0.9320
Precision: 0.9667
Recall: 0.9427
F1 Score: 0.9545


In [14]:
for actual,pred in zip(y, predictions):
    print(f'Actual: {actual}, pred: {pred}') 

Actual: 1, pred: [0.90358937]
Actual: 0, pred: [0.0002831]
Actual: 1, pred: [0.8676358]
Actual: 1, pred: [0.9905067]
Actual: 0, pred: [0.00178548]
Actual: 0, pred: [0.02222159]
Actual: 1, pred: [0.70120203]
Actual: 1, pred: [0.9999028]
Actual: 1, pred: [0.99969393]
Actual: 1, pred: [0.9987575]
Actual: 1, pred: [0.8862793]
Actual: 0, pred: [0.02387627]
Actual: 1, pred: [0.99785864]
Actual: 1, pred: [0.99964345]
Actual: 1, pred: [0.9990152]
Actual: 0, pred: [0.00257917]
Actual: 1, pred: [0.9999535]
Actual: 1, pred: [0.9999786]
Actual: 1, pred: [0.9997968]
Actual: 1, pred: [0.99945277]
Actual: 1, pred: [0.97161406]
Actual: 1, pred: [0.99740505]
Actual: 1, pred: [0.826664]
Actual: 1, pred: [0.46393046]
Actual: 1, pred: [0.99331206]
Actual: 1, pred: [0.9348779]
Actual: 1, pred: [0.9999908]
Actual: 1, pred: [0.98969984]
Actual: 1, pred: [0.998966]
Actual: 1, pred: [0.77005184]
Actual: 1, pred: [0.99859494]
Actual: 1, pred: [0.5016059]
Actual: 1, pred: [0.9999439]
Actual: 1, pred: [0.99843264